In [6]:
import os
import sys
import argparse
from Bio.PDB import PDBParser
from Bio.SeqUtils import seq1

STANDARD_AA = set("ACDEFGHIKLMNPQRSTVWY")
TARGET_AA = "A"


def generate_ddg_mutfiles(input_pdb: str, output_dir: str, chain_id: str = "L"):
    parser = PDBParser(QUIET=True)
    structure = parser.get_structure("protein", input_pdb)
    model = structure[0]

    if chain_id not in model:
        raise KeyError(f"Chain {chain_id} not found in {input_pdb}")

    chain = model[chain_id]
    os.makedirs(output_dir, exist_ok=True)

    pose_index = 0
    for model_chain in model:
        for residue in model_chain:
            hetflag, _, _ = residue.get_id()
            if hetflag.strip():
                continue

            before_mut_res = seq1(residue.get_resname())
            if before_mut_res not in STANDARD_AA:
                continue

            pose_index += 1
            if model_chain.id != chain_id or before_mut_res == TARGET_AA:
                continue

            mutfile_path = os.path.join(output_dir, f"{pose_index}{TARGET_AA}.mutfile")
            with open(mutfile_path, "w") as file:
                file.write("total 1\n")
                file.write("1\n")
                file.write(f"{before_mut_res} {pose_index} {TARGET_AA}\n")


def gen_joblist(input_pdb: str, mutfiles_dir: str, output_dir: str, joblist_path: str):
    with open(joblist_path, "w") as joblist:
        for mutfile in sorted(os.listdir(mutfiles_dir)):
            if mutfile.endswith(".mutfile"):
                mutfile_path = os.path.abspath(os.path.join(mutfiles_dir, mutfile))
                joblist.write(
                    f"cartesian_ddg.mpi.linuxgccrelease -s {input_pdb} -ddg::mut_file {mutfile_path} "
                    f"-relax:min_type lbfgs_armijo_nonmonotone -ex1 -ex2 -use_input_sc -flip_HNQ "
                    f"-fa_max_dis 9.0 -ddg:iterations 3 -ddg::dump_pdbs false -bbnbrs 1 -score:weights beta_jan25_cart -interface_ddg 1 "
                    f"-optimization:default_max_cycles 200 -crystal_refine -relax:cartesian -beta_jan25_cart "
                    f"-out:path:all {output_dir} > {os.path.join(output_dir, f'{mutfile}.log')} 2>&1\n"
                )


def iter_pdb_files(input_path: str):
    if os.path.isfile(input_path):
        if input_path.lower().endswith(".pdb"):
            return [input_path]
        raise ValueError(f"Input file is not a pdb: {input_path}")

    if os.path.isdir(input_path):
        pdb_files = [
            os.path.join(input_path, name)
            for name in sorted(os.listdir(input_path))
            if name.lower().endswith(".pdb")
        ]
        if not pdb_files:
            raise FileNotFoundError(f"No pdb files found in {input_path}")
        return pdb_files

    raise FileNotFoundError(f"Input path does not exist: {input_path}")


def process_pdb(input_pdb: str, output_root: str, chain_id: str = "L"):
    pdb_name = os.path.splitext(os.path.basename(input_pdb))[0]
    pdb_root = os.path.join(output_root, f"ddg_{pdb_name}")
    mutfiles_dir = os.path.join(pdb_root, "mutfiles")
    ddg_out_dir = os.path.join(pdb_root, "ddg_out")

    os.makedirs(mutfiles_dir, exist_ok=True)
    os.makedirs(ddg_out_dir, exist_ok=True)

    generate_ddg_mutfiles(input_pdb, mutfiles_dir, chain_id)
    joblist_path = os.path.join(pdb_root, "job.list")
    gen_joblist(input_pdb, mutfiles_dir, ddg_out_dir, joblist_path)


if __name__ == "__main__" and "ipykernel" not in sys.modules:
    arg_parser = argparse.ArgumentParser(description="Generate alanine-scan ddg mutfiles from PDB files.")
    arg_parser.add_argument("-i", "--input", required=True, help="PDB文件路径或包含多个PDB文件的目录")
    arg_parser.add_argument("-o", "--output", default="ddg_files", help="总输出目录，默认ddg_files")
    arg_parser.add_argument("-c", "--chain", default="L", help="链ID，默认L")
    args = arg_parser.parse_args()

    for input_pdb in iter_pdb_files(args.input):
        process_pdb(input_pdb, args.output, args.chain)

In [7]:
for input_pdb in iter_pdb_files("/home/junjiechen/1_work/250401-Dpepalign/Benchmark/RFdiffusion2/minimized"):
    process_pdb(input_pdb, "/home/junjiechen/1_work/250401-Dpepalign/Benchmark/RFdiffusion2/alanine-scan/ddg_files")

In [2]:
import os
for dir in os.listdir("./ddg_files_rest"):
    if dir in os.listdir("./ddg_files"):
        ddg_rest_out_path = os.path.join("./ddg_files_rest", dir, "ddg_out")
        ddg_out_path = os.path.join("./ddg_files", dir, "ddg_out")
        print(ddg_rest_out_path)
        print(ddg_out_path)
        os.system(f"cp -r {ddg_rest_out_path}/* {ddg_out_path}/")

./ddg_files_rest/ddg_5xxq_0019/ddg_out
./ddg_files/ddg_5xxq_0019/ddg_out
./ddg_files_rest/ddg_5gu4_0003/ddg_out
./ddg_files/ddg_5gu4_0003/ddg_out
./ddg_files_rest/ddg_6f0w_0007/ddg_out
./ddg_files/ddg_6f0w_0007/ddg_out
./ddg_files_rest/ddg_5ggp_0009/ddg_out
./ddg_files/ddg_5ggp_0009/ddg_out
./ddg_files_rest/ddg_5v2p_0018/ddg_out
./ddg_files/ddg_5v2p_0018/ddg_out
./ddg_files_rest/ddg_5n22_0001/ddg_out
./ddg_files/ddg_5n22_0001/ddg_out
./ddg_files_rest/ddg_6g5g_0004/ddg_out
./ddg_files/ddg_6g5g_0004/ddg_out
./ddg_files_rest/ddg_6f6d_0002/ddg_out
./ddg_files/ddg_6f6d_0002/ddg_out
./ddg_files_rest/ddg_6fbk_0001/ddg_out
./ddg_files/ddg_6fbk_0001/ddg_out
./ddg_files_rest/ddg_5lyn_0001/ddg_out
./ddg_files/ddg_5lyn_0001/ddg_out
./ddg_files_rest/ddg_5yay_0001/ddg_out
./ddg_files/ddg_5yay_0001/ddg_out
./ddg_files_rest/ddg_5onp_0001/ddg_out
./ddg_files/ddg_5onp_0001/ddg_out
./ddg_files_rest/ddg_5fv6_0010/ddg_out
./ddg_files/ddg_5fv6_0010/ddg_out
./ddg_files_rest/ddg_5t0k_0001/ddg_out
./ddg_files/

In [1]:
# chech completeness
import os

def collect_completed_ids(status_dir: str, index: int = 1) -> set[str]:
    completed = set()
    for file in os.listdir(status_dir):
        if not file.endswith(".ok"):
            continue
        parts = os.path.splitext(file)[0].split("_")
        if len(parts) > index:
            completed.add(parts[index])
    return completed


comp1 = collect_completed_ids(".alascan_status_20260403_223931", index=1)
comp2 = collect_completed_ids(".alascan_status_20260405_203359", index=1)
comp = comp1 | comp2

print(f"comp1 count: {len(comp1)}")
print(f"comp2 count: {len(comp2)}")
print(f"union count: {len(comp)}")

minimized_dir = "/home/junjiechen/1_work/250401-Dpepalign/Benchmark/RFdiffusion2/minimized"
minimized_ids = {
    os.path.splitext(file)[0].split("_")[0]
    for file in os.listdir(minimized_dir)
    if file.lower().endswith(".pdb")
}

missing = sorted(minimized_ids - comp)
extra = sorted(comp - minimized_ids)

print(f"minimized count: {len(minimized_ids)}")
print(f"missing count: {len(missing)}")
print(f"extra count: {len(extra)}")

if missing:
    print("Missing IDs:")
    print(missing)
else:
    print("All minimized entries are covered by comp1 + comp2.")

comp1 count: 133
comp2 count: 37
union count: 170
minimized count: 170
missing count: 0
extra count: 0
All minimized entries are covered by comp1 + comp2.
